In [0]:
print('helpers')

In [0]:
import requests
import json

In [0]:
CATALOG_NAME = 'iot_sensor_catalog'
Default_schema = 'default'
Volume_name = 'iot_sensor_volume'
CATALOG_VOLUME = f'{CATALOG_NAME}.{Default_schema}.{Volume_name}'
Bronze_schema = 'Bronze_IOT'
BRONZE_DATABASE = f'{CATALOG_NAME}.{Bronze_schema}'


In [0]:
class DatabricksJobManager:
    def __init__(self, job_name, job_settings):
        self.job_name = job_name
        self.job_settings = job_settings
        self.api_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
        self.api_token =dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        self.headers = {
            "Authorization": f"Bearer {self.api_token}",
            "Content-Type": "application/json"
        }

    def create_job_if_not_exists(self):
        list_jobs_response = requests.get(
            f"{self.api_url}/api/2.1/jobs/list",
            headers=self.headers
        )
        jobs = list_jobs_response.json().get("jobs", [])
        job_exists = [i['settings']['name'] for i in jobs]

        if self.job_name in job_exists:
            print(f"Job {self.job_name} already exists.")
        else:
            response = requests.post(
                f"{self.api_url}/api/2.1/jobs/create",
                headers=self.headers,
                data=json.dumps(self.job_settings)
            )
            job_response = response.json()

            print('job created successfully')
    


In [0]:
class DatabricksPipelineManager:
    def __init__(self, pipeline_name, pipeline_payload):
        self.pipeline_name = pipeline_name
        self.pipeline_settings = pipeline_payload
        self.api_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
        self.TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        
    def create_pipeline_if_not_exists(self):
        # Check if pipeline already exists
        list_response = requests.get(
            f"{self.api_url}/api/2.0/pipelines",
            headers={
                "Authorization": f"Bearer {self.TOKEN}"
            }
        )
        pipelines = list_response.json().get("statuses", [])
        pipeline_exists = [i['name'] for i in pipelines]
        if pipeline_name in pipeline_exists:
            print(f"Pipeline {pipeline_name} already exists")
        else:
            # Create the pipeline
            response = requests.post(
                f"{self.api_url}/api/2.0/pipelines",
                headers={
                    "Authorization": f"Bearer {self.TOKEN}",
                    "Content-Type": "application/json"
                },
                json=pipeline_payload
            )

            print('pipeline created successfully')